# CNN-EXP-045 — WIDTH512, 50k, two-phase LR schedule

## Цель

Проверяем следующий шаг capacity scaling:

```text
128 → 256 → 512 channels
```

Для WIDTH512 сразу используем режим обучения, который оказался более разумным
для широкой модели:

```text
0 → 25k:
    LR = 1e-3

после 25k:
    LR jump → 3e-4

25k → 50k:
    CosineAnnealingLR
    3e-4 → 3e-5
```

Архитектурно меняется только width:

```text
channels = 512
RF = 1029
те же 8 residual blocks
те же targets
тот же loss
тот же sampler
тот же split
batch = 4
```

## Важно: это LONG / OVERNIGHT RUN

Ориентировочный runtime на текущем DirectML железе:

```text
training + final strong validation:
примерно 6–12 часов
```

Это грубая оценка. WIDTH512 имеет примерно ×4 convolution FLOPs относительно WIDTH256,
а DirectML runtime может масштабироваться нелинейно.

Ноутбук печатает текущий `sec/step` и ETA во время обучения.

### Главное сравнение после запуска

```text
WIDTH256 @50k с хорошим LR schedule
vs
WIDTH512 @50k с тем же двухфазным schedule
```

Промежуточные checkpoints сохраняются, но **полный holdout на каждом checkpoint НЕ запускается**.
Full strong validation выполняется только один раз — на финальном 50k.


In [1]:
from pathlib import Path
import copy
import gc
import json
import math
import sys
import time
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

import torch
import torch_directml

from scipy.stats import pearsonr, rankdata

CURRENT_DIR = Path.cwd().resolve()

PROJECT_ROOT = next(
    candidate
    for candidate in (CURRENT_DIR, *CURRENT_DIR.parents)
    if (candidate / "README.md").is_file()
    and (candidate / "src/ml_project").is_dir()
)

SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from ml_project.epic_baseline import load_baseline_sequences
from ml_project.epic_cnn import (
    DirectMLAdam,
    ProfileWindowConfig,
    ProfileWindowGenerator,
    create_cnn_presence_intensity,
    dml_weighted_huber,
    dml_weighted_profile_bce_with_logits,
    forward_both_strands_multitask,
)
from ml_project.epic_data import load_split

print("Project:", PROJECT_ROOT)


Project: D:\Projects\EPIC\EPIC


In [2]:
device = torch_directml.device()

print("DirectML device:", device)
print("GPU:", torch_directml.device_name(0))


DirectML device: privateuseone:0
GPU: AMD Radeon RX 6750 GRE 12GB 


In [3]:
# ============================================================
# DATA — frozen relative to reference
# ============================================================

split = load_split(PROJECT_ROOT)
sequences, fasta_alphabet = load_baseline_sequences(split)

train_contigs = set(
    split.intervals("train")["contig"]
)

validation_contigs = set(
    split.intervals("validation")["contig"]
)

assert train_contigs.isdisjoint(
    validation_contigs
)

display(split.summary)

print("FASTA contigs:", len(sequences))
print("Train contigs:", len(train_contigs))
print("Validation contigs:", len(validation_contigs))

profile_config = ProfileWindowConfig(
    target_length=1024,
    context_flank=130,
    positive_branch_fraction=0.5,
)

train_generator = ProfileWindowGenerator(
    prepared_split=split,
    sequences=sequences,
    part="train",
    config=profile_config,
    seed=42,
)

train_batch = train_generator.sample_batch(
    batch_size=4
)

print()
print("Input:", train_batch.x.shape)
print("Both orientations:", train_batch.x_both_strands.shape)
print("Target:", train_batch.target.shape)
print("Count:", train_batch.count.shape)
print("Mask:", train_batch.mask.shape)
print("Positive position-strand:", int(train_batch.target.sum()))


,split,contigs,intervals_both_strands,whitelist_bp,position_strand_count,percent_of_all_positions,positive_positions,negative_positions,positive_percent
0,train,9,168998,114032080,228064160,59.316784,170443.0,227893717.0,0.074735
1,validation,3,60386,41903743,83807486,21.797334,56205.0,83751281.0,0.067064
2,test,3,54846,36306695,72613390,18.885882,NaN,NaN,NaN


FASTA contigs: 47
Train contigs: 9
Validation contigs: 3

Input: torch.Size([4, 5, 1284])
Both orientations: torch.Size([8, 5, 1284])
Target: torch.Size([4, 2, 1024])
Count: torch.Size([4, 2, 1024])
Mask: torch.Size([4, 2, 1024])
Positive position-strand: 33


## Проверка архитектуры

Ожидаем:

| Stage | Channels | RF |
|---|---:|---:|
| Stem | 5 → 512 | 1 |
| Block 1, k7 d1 | 512 → 512 | 13 |
| Block 2, k3 d2 | 512 → 512 | 21 |
| Block 3, k3 d4 | 512 → 512 | 37 |
| Block 4, k3 d8 | 512 → 512 | 69 |
| Block 5, k3 d16 | 512 → 512 | 133 |
| Block 6, k3 d32 | 512 → 512 | 261 |
| Block 7, k3 d64 | 512 → 512 | 517 |
| Block 8, k3 d128 | 512 → 512 | **1029** |
| Profile head | 512 → 2 | 1029 |

Ожидаемое число trainable parameters:

```text
14,708,738
```

Формула для этой архитектуры:

```text
params(W) = 56·W² + 56·W + 2
```

При `W=512` получаем `14,708,738`.


In [4]:
# ============================================================
# ARCHITECTURE ASSERT
# ============================================================

EXPECTED_CHANNELS = 512
EXPECTED_RF = 1029
EXPECTED_PARAMETER_COUNT = 14_708_738

architecture_model = create_cnn_presence_intensity(channels=512)

stem = architecture_model.backbone.stem

actual_channels = int(
    stem.out_channels
)

parameter_count = sum(
    p.numel()
    for p in architecture_model.parameters()
    if p.requires_grad
)

# Compute theoretical RF from actual residual blocks.
rf = 1

for block in architecture_model.backbone.blocks:
    for conv in (block.conv1, block.conv2):
        k = int(conv.kernel_size[0])
        d = int(conv.dilation[0])

        rf += (
            k - 1
        ) * d

print("Stem:", stem.in_channels, "→", stem.out_channels)
print("Residual blocks:", len(architecture_model.backbone.blocks))
print("Theoretical RF:", rf)
print("Profile head:", architecture_model.profile_head.in_channels, "→", architecture_model.profile_head.out_channels)
print("Trainable parameters:", f"{parameter_count:,}")

assert actual_channels == EXPECTED_CHANNELS, (
    f"Сейчас model.py всё ещё создаёт {actual_channels} channels, "
    f"а нужен {EXPECTED_CHANNELS}. "
    "Проверь default channels в create_cnn_baseline() "
    "и create_cnn_presence_intensity(channels=512)."
)

assert rf == EXPECTED_RF
assert architecture_model.profile_head.in_channels == EXPECTED_CHANNELS
assert architecture_model.profile_head.out_channels == 2
assert parameter_count == EXPECTED_PARAMETER_COUNT

del architecture_model
gc.collect()

print()
print("ARCHITECTURE CHECK: OK")


Stem: 5 → 512
Residual blocks: 8
Theoretical RF: 1029
Profile head: 512 → 2
Trainable parameters: 14,708,738

ARCHITECTURE CHECK: OK


In [5]:
# ============================================================
# ONE-STEP DIRECTML / GRADIENT SMOKE
# ============================================================

smoke_model = (
    create_cnn_presence_intensity(channels=512)
    .to(device)
)

smoke_model.train()

smoke_optimizer = DirectMLAdam(
    smoke_model.parameters(),
    lr=1e-3,
)

smoke_batch = train_generator.sample_batch(
    batch_size=4
)

x_gpu = smoke_batch.x_both_strands.to(
    device
)

plus_target_gpu = (
    smoke_batch.plus_target
    .to(device)
)

minus_target_reverse_gpu = (
    smoke_batch.minus_target_reverse
    .to(device)
)

plus_presence_weight_gpu = (
    smoke_batch.plus_loss_weight
    .to(device)
)

minus_presence_weight_reverse_gpu = (
    smoke_batch.minus_loss_weight_reverse
    .to(device)
)

plus_count = (
    smoke_batch.count[:, 0:1, :]
    .contiguous()
)

minus_count_reverse = (
    smoke_batch.count[:, 1:2, :]
    .flip(-1)
    .contiguous()
)

plus_intensity_target_gpu = (
    torch.log1p(plus_count)
    .to(device)
)

minus_intensity_target_reverse_gpu = (
    torch.log1p(minus_count_reverse)
    .to(device)
)

plus_intensity_weight_gpu = (
    2.0
    * smoke_batch.plus_loss_weight
    * (plus_count > 0).float()
).to(device)

minus_intensity_weight_reverse_gpu = (
    2.0
    * smoke_batch.minus_loss_weight_reverse
    * (minus_count_reverse > 0).float()
).to(device)

smoke_optimizer.zero_grad(
    set_to_none=True
)

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")

    (
        plus_presence,
        minus_presence_reverse,
        plus_intensity,
        minus_intensity_reverse,
    ) = forward_both_strands_multitask(
        smoke_model,
        x_gpu,
        batch_size=4,
        context_flank=profile_config.context_flank,
        target_length=profile_config.target_length,
    )

    presence_loss = (
        dml_weighted_profile_bce_with_logits(
            plus_presence,
            minus_presence_reverse,
            plus_target_gpu,
            minus_target_reverse_gpu,
            plus_presence_weight_gpu,
            minus_presence_weight_reverse_gpu,
        )
    )

    intensity_loss = (
        dml_weighted_huber(
            plus_intensity,
            plus_intensity_target_gpu,
            plus_intensity_weight_gpu,
        )
        + dml_weighted_huber(
            minus_intensity_reverse,
            minus_intensity_target_reverse_gpu,
            minus_intensity_weight_reverse_gpu,
        )
    )

    smoke_loss = (
        presence_loss
        + 0.1 * intensity_loss
    )

    smoke_loss.backward()
    smoke_optimizer.step()

fallback_warnings = [
    str(item.message)
    for item in caught
    if "fall back" in str(item.message).lower()
]

presence_gradient = float(
    smoke_model.profile_head.weight.grad[0]
    .abs()
    .sum()
    .detach()
    .cpu()
    .item()
)

intensity_gradient = float(
    smoke_model.profile_head.weight.grad[1]
    .abs()
    .sum()
    .detach()
    .cpu()
    .item()
)

print("Presence loss:", float(presence_loss.detach().cpu().item()))
print("Intensity loss:", float(intensity_loss.detach().cpu().item()))
print("Presence gradient:", presence_gradient)
print("Intensity gradient:", intensity_gradient)
print("CPU fallback:", fallback_warnings)

assert presence_gradient > 0
assert intensity_gradient > 0
assert not fallback_warnings

del (
    smoke_model,
    smoke_optimizer,
    smoke_batch,
    x_gpu,
    plus_target_gpu,
    minus_target_reverse_gpu,
    plus_presence_weight_gpu,
    minus_presence_weight_reverse_gpu,
    plus_intensity_target_gpu,
    minus_intensity_target_reverse_gpu,
    plus_intensity_weight_gpu,
    minus_intensity_weight_reverse_gpu,
    plus_presence,
    minus_presence_reverse,
    plus_intensity,
    minus_intensity_reverse,
    presence_loss,
    intensity_loss,
    smoke_loss,
)

gc.collect()

print("DIRECTML SMOKE: OK")


Presence loss: 0.7384037971496582
Intensity loss: 1.122327208518982
Presence gradient: 48.479251861572266
Intensity gradient: 26.80286407470703
CPU fallback: []
DIRECTML SMOKE: OK


## Обучение — overnight режим

Протокол:

```text
steps = 50,000
batch = 4
seed = 42
intensity weight = 0.1
```

LR:

```text
1 → 25,000:
    1e-3 constant

после step 25,000:
    jump 1e-3 → 3e-4

25,001 → 50,000:
    cosine 3e-4 → 3e-5
```

Checkpoints:

```text
10k
25k
30k
35k
40k
45k
50k
```

**Никаких промежуточных full-holdout validation.**

Сначала выполняется DirectML smoke-test на `batch=4`.
Если WIDTH512 не помещается в память, ошибка появится сразу, а не спустя часы.
В этом случае batch автоматически НЕ уменьшаем, потому что BatchNorm сделает сравнение уже другим экспериментом.


In [6]:
# ============================================================
# EXPERIMENT CONFIG
# ============================================================

EXPERIMENT_ID = "CNN-EXP-045"
EXPERIMENT_SLUG = "CNN_EXP045_WIDTH512_50K_TWO_PHASE_LR_RF1029"

SEED = 42

STEPS = 50_000
PHASE1_END_STEP = 25_000

BATCH_SIZE = 4
LOG_EVERY = 250

CHECKPOINT_STEPS = {
    10_000,
    25_000,
    30_000,
    35_000,
    40_000,
    45_000,
    50_000,
}

PHASE1_LR = 1e-3
PHASE2_START_LR = 3e-4
PHASE2_END_LR = 3e-5

PHASE2_STEPS = (
    STEPS
    - PHASE1_END_STEP
)

INTENSITY_LOSS_WEIGHT = 0.1

RUN_DIR = (
    PROJECT_ROOT
    / "artifacts"
    / "experiments"
    / EXPERIMENT_SLUG
    / "run_001"
)

RUN_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

FINAL_CHECKPOINT = (
    RUN_DIR
    / "width512_50k_final.pt"
)

HISTORY_PATH = (
    RUN_DIR
    / "training_history.csv"
)

print("Experiment:", EXPERIMENT_ID)
print("Artifacts:", RUN_DIR)
print("Channels:", EXPECTED_CHANNELS)
print("RF:", EXPECTED_RF)
print("Steps:", STEPS)
print("Batch:", BATCH_SIZE)
print("Phase 1 LR:", PHASE1_LR)
print(
    "Phase 2 LR:",
    PHASE2_START_LR,
    "→",
    PHASE2_END_LR,
)
print()
print(
    "LONG RUN ESTIMATE: ~6–12 hours "
    "for training + final validation."
)


Experiment: CNN-EXP-045
Artifacts: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP045_WIDTH512_50K_TWO_PHASE_LR_RF1029\run_001
Channels: 512
RF: 1029
Steps: 50000
Batch: 4
Phase 1 LR: 0.001
Phase 2 LR: 0.0003 → 3e-05

LONG RUN ESTIMATE: ~6–12 hours for training + final validation.


In [7]:
# ============================================================
# TRAINING — WIDTH512, TWO-PHASE LR
# ============================================================

torch.manual_seed(SEED)

generator = ProfileWindowGenerator(
    prepared_split=split,
    sequences=sequences,
    part="train",
    config=profile_config,
    seed=SEED,
)

model = (
    create_cnn_presence_intensity(channels=512)
    .to(device)
)

assert (
    model.backbone.stem.out_channels
    == EXPECTED_CHANNELS
)

optimizer = DirectMLAdam(
    model.parameters(),
    lr=PHASE1_LR,
)

scheduler = None


def current_lr():
    return float(
        optimizer.param_groups[0]["lr"]
    )


def model_state_on_cpu(model):
    return {
        name: tensor.detach().cpu().clone()
        for name, tensor in model.state_dict().items()
    }


def tree_to_cpu(value):
    if torch.is_tensor(value):
        return (
            value.detach()
            .cpu()
            .clone()
        )

    if isinstance(value, dict):
        return {
            key: tree_to_cpu(item)
            for key, item in value.items()
        }

    if isinstance(value, list):
        return [
            tree_to_cpu(item)
            for item in value
        ]

    if isinstance(value, tuple):
        return tuple(
            tree_to_cpu(item)
            for item in value
        )

    return value


def checkpoint_path(step):
    return (
        RUN_DIR
        / f"model_step_{step:05d}.pt"
    )


def save_checkpoint(
    path,
    *,
    step,
    elapsed_seconds,
    seen_position_strand,
    history,
):
    checkpoint = {
        "experiment": EXPERIMENT_ID,
        "model_type": "presence_intensity",
        "architecture": "rf1029_width512",
        "step": int(step),

        "model_state_dict": model_state_on_cpu(
            model
        ),

        "optimizer_state_dict": tree_to_cpu(
            optimizer.state_dict()
        ),

        "scheduler_state_dict": (
            tree_to_cpu(
                scheduler.state_dict()
            )
            if scheduler is not None
            else None
        ),

        "generator_rng_state": copy.deepcopy(
            generator.rng.bit_generator.state
        ),

        "torch_rng_state": torch.get_rng_state(),

        "model_config": {
            "input_channels": 5,
            "channels": 512,
            "output_channels": 2,
            "theoretical_rf": 1029,
        },

        "profile_config": {
            "target_length": (
                profile_config.target_length
            ),
            "context_flank": (
                profile_config.context_flank
            ),
            "positive_branch_fraction": (
                profile_config
                .positive_branch_fraction
            ),
        },

        "training_config": {
            "seed": SEED,
            "steps": STEPS,
            "batch_size": BATCH_SIZE,
            "phase1_end_step": PHASE1_END_STEP,
            "phase1_learning_rate": PHASE1_LR,
            "phase2_start_lr": PHASE2_START_LR,
            "phase2_end_lr": PHASE2_END_LR,
            "phase2_scheduler": (
                "CosineAnnealingLR"
            ),
            "phase2_t_max": PHASE2_STEPS,
            "intensity_loss_weight": (
                INTENSITY_LOSS_WEIGHT
            ),
            "huber_delta": 1.0,
        },

        "seen_position_strand": int(
            seen_position_strand
        ),

        "elapsed_seconds": float(
            elapsed_seconds
        ),

        "history": list(history),
    }

    torch.save(
        checkpoint,
        path,
    )


def planned_lr(global_step):
    if global_step <= PHASE1_END_STEP:
        return PHASE1_LR

    t = (
        global_step
        - PHASE1_END_STEP
    )

    return (
        PHASE2_END_LR
        + 0.5
        * (
            PHASE2_START_LR
            - PHASE2_END_LR
        )
        * (
            1.0
            + math.cos(
                math.pi
                * t
                / PHASE2_STEPS
            )
        )
    )


print("Planned LR trajectory:")

for probe_step in (
    1,
    10_000,
    25_000,
    30_000,
    35_000,
    37_500,
    40_000,
    45_000,
    50_000,
):
    print(
        f"step {probe_step:5d}: "
        f"{planned_lr(probe_step):.8f}"
    )

print()

history = []

ema_presence = None
ema_intensity = None
ema_total = None

seen_position_strand = 0

started_at = time.perf_counter()

model.train()

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")

    for step in range(
        1,
        STEPS + 1,
    ):
        batch = generator.sample_batch(
            batch_size=BATCH_SIZE
        )

        x_gpu = (
            batch
            .x_both_strands
            .to(device)
        )

        plus_target_gpu = (
            batch.plus_target
            .to(device)
        )

        minus_target_reverse_gpu = (
            batch.minus_target_reverse
            .to(device)
        )

        plus_presence_weight_gpu = (
            batch.plus_loss_weight
            .to(device)
        )

        minus_presence_weight_reverse_gpu = (
            batch.minus_loss_weight_reverse
            .to(device)
        )

        plus_count = (
            batch.count[:, 0:1, :]
            .contiguous()
        )

        minus_count_reverse = (
            batch.count[:, 1:2, :]
            .flip(-1)
            .contiguous()
        )

        plus_intensity_target_gpu = (
            torch.log1p(
                plus_count
            )
            .to(device)
        )

        minus_intensity_target_reverse_gpu = (
            torch.log1p(
                minus_count_reverse
            )
            .to(device)
        )

        plus_intensity_weight_gpu = (
            2.0
            * batch.plus_loss_weight
            * (plus_count > 0).float()
        ).to(device)

        minus_intensity_weight_reverse_gpu = (
            2.0
            * batch.minus_loss_weight_reverse
            * (minus_count_reverse > 0).float()
        ).to(device)

        optimizer.zero_grad(
            set_to_none=True
        )

        (
            plus_presence,
            minus_presence_reverse,
            plus_intensity,
            minus_intensity_reverse,
        ) = forward_both_strands_multitask(
            model,
            x_gpu,
            batch_size=BATCH_SIZE,
            context_flank=(
                profile_config
                .context_flank
            ),
            target_length=(
                profile_config
                .target_length
            ),
        )

        presence_loss = (
            dml_weighted_profile_bce_with_logits(
                plus_presence,
                minus_presence_reverse,
                plus_target_gpu,
                minus_target_reverse_gpu,
                plus_presence_weight_gpu,
                minus_presence_weight_reverse_gpu,
            )
        )

        intensity_loss = (
            dml_weighted_huber(
                plus_intensity,
                plus_intensity_target_gpu,
                plus_intensity_weight_gpu,
                delta=1.0,
            )
            + dml_weighted_huber(
                minus_intensity_reverse,
                minus_intensity_target_reverse_gpu,
                minus_intensity_weight_reverse_gpu,
                delta=1.0,
            )
        )

        total_loss = (
            presence_loss
            + INTENSITY_LOSS_WEIGHT
            * intensity_loss
        )

        presence_value = float(
            presence_loss
            .detach()
            .cpu()
            .item()
        )

        intensity_value = float(
            intensity_loss
            .detach()
            .cpu()
            .item()
        )

        total_value = float(
            total_loss
            .detach()
            .cpu()
            .item()
        )

        if not all(
            math.isfinite(value)
            for value in (
                presence_value,
                intensity_value,
                total_value,
            )
        ):
            raise FloatingPointError(
                f"Non-finite loss at step {step}"
            )

        total_loss.backward()
        optimizer.step()

        # Exact branch point:
        # step 25k itself is completed at LR=1e-3.
        # Only AFTER that step do we switch to phase 2.
        if step == PHASE1_END_STEP:

            # Save the exact branch point before LR mutation.
            elapsed = (
                time.perf_counter()
                - started_at
            )

            if ema_presence is None:
                branch_history = []
            else:
                branch_history = history

            save_checkpoint(
                checkpoint_path(step),
                step=step,
                elapsed_seconds=elapsed,
                seen_position_strand=(
                    seen_position_strand
                    + int(
                        batch.mask.sum()
                    )
                ),
                history=branch_history,
            )

            for group in optimizer.param_groups:
                group["lr"] = (
                    PHASE2_START_LR
                )
                group["initial_lr"] = (
                    PHASE2_START_LR
                )

            scheduler = (
                torch.optim.lr_scheduler
                .CosineAnnealingLR(
                    optimizer,
                    T_max=PHASE2_STEPS,
                    eta_min=PHASE2_END_LR,
                )
            )

        elif step > PHASE1_END_STEP:
            scheduler.step()

        lr_value = current_lr()

        if ema_presence is None:
            ema_presence = presence_value
            ema_intensity = intensity_value
            ema_total = total_value

        else:
            ema_presence = (
                0.99 * ema_presence
                + 0.01 * presence_value
            )

            ema_intensity = (
                0.99 * ema_intensity
                + 0.01 * intensity_value
            )

            ema_total = (
                0.99 * ema_total
                + 0.01 * total_value
            )

        seen_position_strand += int(
            batch.mask.sum()
        )

        elapsed = (
            time.perf_counter()
            - started_at
        )

        if (
            step % LOG_EVERY == 0
            or step == STEPS
        ):
            seconds_per_step = (
                elapsed
                / step
            )

            remaining_training_seconds = (
                STEPS
                - step
            ) * seconds_per_step

            record = {
                "step": int(step),
                "learning_rate": float(
                    lr_value
                ),
                "presence_loss": presence_value,
                "intensity_loss": intensity_value,
                "total_loss": total_value,
                "ema_presence": float(
                    ema_presence
                ),
                "ema_intensity": float(
                    ema_intensity
                ),
                "ema_total": float(
                    ema_total
                ),
                "elapsed_seconds": float(
                    elapsed
                ),
                "seconds_per_step": float(
                    seconds_per_step
                ),
                "estimated_training_hours_remaining": float(
                    remaining_training_seconds
                    / 3600.0
                ),
                "seen_position_strand": int(
                    seen_position_strand
                ),
                "positions_per_second": float(
                    seen_position_strand
                    / elapsed
                ),
            }

            history.append(
                record
            )

            print(
                f"step={step:5d} "
                f"lr={lr_value:.8f} "
                f"presence={presence_value:.4f} "
                f"intensity={intensity_value:.4f} "
                f"total={total_value:.4f} "
                f"ema=({ema_presence:.4f}, "
                f"{ema_intensity:.4f}, "
                f"{ema_total:.4f}) "
                f"sec/step={seconds_per_step:.3f} "
                f"ETA_train="
                f"{remaining_training_seconds / 3600.0:.2f}h"
            )

        # 25k was already saved immediately before phase switch.
        if (
            step in CHECKPOINT_STEPS
            and step != PHASE1_END_STEP
        ):
            save_checkpoint(
                checkpoint_path(step),
                step=step,
                elapsed_seconds=elapsed,
                seen_position_strand=(
                    seen_position_strand
                ),
                history=history,
            )

fallback_warnings = [
    str(item.message)
    for item in caught
    if "fall back" in str(item.message).lower()
]

elapsed = (
    time.perf_counter()
    - started_at
)

save_checkpoint(
    FINAL_CHECKPOINT,
    step=STEPS,
    elapsed_seconds=elapsed,
    seen_position_strand=seen_position_strand,
    history=history,
)

pd.DataFrame(
    history
).to_csv(
    HISTORY_PATH,
    index=False,
)

print()
print("=" * 72)
print("TRAINING COMPLETE")
print("=" * 72)
print("Training hours:", round(elapsed / 3600.0, 3))
print("Final LR:", current_lr())
print("Target final LR:", PHASE2_END_LR)
print("Final EMA presence:", round(ema_presence, 6))
print("Final EMA intensity:", round(ema_intensity, 6))
print("Final EMA total:", round(ema_total, 6))
print("CPU fallback:", fallback_warnings)
print("Checkpoint:", FINAL_CHECKPOINT)
print("History:", HISTORY_PATH)

assert scheduler is not None

assert math.isclose(
    current_lr(),
    PHASE2_END_LR,
    rel_tol=1e-6,
    abs_tol=1e-10,
)

assert not fallback_warnings
assert FINAL_CHECKPOINT.is_file()
assert HISTORY_PATH.is_file()


Planned LR trajectory:
step     1: 0.00100000
step 10000: 0.00100000
step 25000: 0.00100000
step 30000: 0.00027422
step 35000: 0.00020672
step 37500: 0.00016500
step 40000: 0.00012328
step 45000: 0.00005578
step 50000: 0.00003000

step=  250 lr=0.00100000 presence=0.4969 intensity=0.4126 total=0.5382 ema=(0.6574, 0.6622, 0.7236) sec/step=0.235 ETA_train=3.25h
step=  500 lr=0.00100000 presence=0.5421 intensity=0.0216 total=0.5443 ema=(0.5301, 0.2879, 0.5589) sec/step=0.232 ETA_train=3.20h
step=  750 lr=0.00100000 presence=0.5551 intensity=0.2997 total=0.5851 ema=(0.5325, 0.2988, 0.5623) sec/step=0.231 ETA_train=3.17h
step= 1000 lr=0.00100000 presence=0.4087 intensity=0.2061 total=0.4293 ema=(0.5269, 0.3185, 0.5588) sec/step=0.231 ETA_train=3.14h
step= 1250 lr=0.00100000 presence=0.3474 intensity=0.8796 total=0.4354 ema=(0.5285, 0.3352, 0.5620) sec/step=0.231 ETA_train=3.12h
step= 1500 lr=0.00100000 presence=0.3298 intensity=0.2363 total=0.3535 ema=(0.4985, 0.2707, 0.5255) sec/step=0.231

# Сильная validation — только финальный WIDTH512 @50k

После overnight training выполняется **один** полный natural-background holdout.

Считаем:
- Presence AP 5dp;
- Presence EPIC Spearman raw / 5dp;
- Intensity EPIC Spearman raw;
- per-contig;
- HIGH_TP / LOW_TP / HARD_FP / random TN;
- frozen feature probes;
- LOW_TP score-matched probe.

Промежуточные checkpoints сохранены, но автоматически НЕ валидируются.
Если понадобится проверить 35k/40k/45k — сделаем отдельным коротким evaluator notebook.


In [8]:
# ============================================================
# LOAD 10K CHECKPOINT
# ============================================================

checkpoint = torch.load(
    FINAL_CHECKPOINT,
    map_location="cpu",
    weights_only=False,
)

validation_model = (
    create_cnn_presence_intensity(channels=512)
)

validation_model.load_state_dict(
    checkpoint[
        "model_state_dict"
    ]
)

assert (
    validation_model
    .backbone
    .stem
    .out_channels
    == EXPECTED_CHANNELS
)

validation_model = validation_model.to(
    device
)

validation_model.eval()

validation_generator = ProfileWindowGenerator(
    prepared_split=split,
    sequences=sequences,
    part="validation",
    config=profile_config,
    seed=42,
)

validation_tiles = (
    validation_generator
    .tile_index
    .copy()
)

print("Checkpoint step:", checkpoint["step"])
print("Validation tiles:", len(validation_tiles))
print(
    "Validation position-strand:",
    int(
        validation_tiles[
            "allowed_position_strand"
        ].sum()
    ),
)
print(
    "Validation positives:",
    validation_generator.total_positive,
)

del checkpoint
gc.collect()


Checkpoint step: 50000
Validation tiles: 47743
Validation position-strand: 83807486
Validation positives: 56205


1390

In [9]:
# ============================================================
# METRIC HELPERS
# ============================================================

SUBMISSION_LEVELS = 100_001


def sigmoid_numpy(x):
    x = np.asarray(
        x,
        dtype=np.float64,
    )

    out = np.empty_like(
        x,
        dtype=np.float64,
    )

    positive = (
        x >= 0
    )

    out[
        positive
    ] = (
        1.0
        / (
            1.0
            + np.exp(
                -x[
                    positive
                ]
            )
        )
    )

    exp_x = np.exp(
        x[
            ~positive
        ]
    )

    out[
        ~positive
    ] = (
        exp_x
        / (
            1.0
            + exp_x
        )
    )

    return out.astype(
        np.float32
    )


def probability_bin_5dp(
    probabilities,
):
    return np.rint(
        np.clip(
            probabilities,
            0.0,
            1.0,
        )
        * 100_000
    ).astype(
        np.int32
    )


def update_histogram(
    hist,
    indices,
):
    values, counts = np.unique(
        indices,
        return_counts=True,
    )

    hist[
        values
    ] += counts.astype(
        np.int64
    )


def ap_from_histograms(
    positive_hist,
    total_hist,
):
    positives = int(
        positive_hist.sum()
    )

    if positives <= 0:
        return float("nan")

    pos_desc = (
        positive_hist[
            ::-1
        ]
    )

    total_desc = (
        total_hist[
            ::-1
        ]
    )

    tp = np.cumsum(
        pos_desc,
        dtype=np.int64,
    )

    predicted_positive = np.cumsum(
        total_desc,
        dtype=np.int64,
    )

    precision = (
        tp
        / np.maximum(
            predicted_positive,
            1,
        )
    )

    return float(
        np.sum(
            (
                pos_desc
                / positives
            )
            * precision
        )
    )


def dense_rank_pearson(
    prediction,
    target,
):
    prediction = np.asarray(
        prediction
    )

    target = np.asarray(
        target
    )

    if len(prediction) < 2:
        return float("nan")

    pred_rank = rankdata(
        prediction,
        method="dense",
    )

    target_rank = rankdata(
        target,
        method="dense",
    )

    return float(
        pearsonr(
            pred_rank,
            target_rank,
        )[0]
    )


def new_segment_state():
    return {
        "total_hist": np.zeros(
            SUBMISSION_LEVELS,
            dtype=np.int64,
        ),
        "positive_hist": np.zeros(
            SUBMISSION_LEVELS,
            dtype=np.int64,
        ),
        "positive_presence_raw": [],
        "positive_presence_5dp": [],
        "positive_intensity": [],
        "positive_count": [],
        "random_negative_score": [],
        "hard_negative_score": [],
    }


In [10]:
# ============================================================
# MEMORY-SAFE FULL HOLDOUT
# ============================================================

EVAL_BATCH_SIZE = 1
PROGRESS_EVERY = 100

RANDOM_NEG_PER_TILE_STRAND = 2
HARD_NEG_PER_TILE_STRAND = 2

rng = np.random.default_rng(
    20260924
)

segment_names = [
    "overall",
    *sorted(
        validation_tiles[
            "contig"
        ].astype(str).unique()
    ),
]

states = {
    segment: new_segment_state()
    for segment in segment_names
}

positions_seen = 0
started_eval = time.perf_counter()

validation_model.eval()

with torch.no_grad():

    for batch_no, start in enumerate(
        range(
            0,
            len(validation_tiles),
            EVAL_BATCH_SIZE,
        ),
        1,
    ):
        selected = (
            validation_tiles
            .iloc[
                start:
                start + EVAL_BATCH_SIZE
            ]
        )

        batch = (
            validation_generator
            .make_batch(
                selected
            )
        )

        B = len(selected)

        x_gpu = (
            batch
            .x_both_strands
            .to(device)
        )

        (
            plus_presence_gpu,
            minus_presence_reverse_gpu,
            plus_intensity_gpu,
            minus_intensity_reverse_gpu,
        ) = forward_both_strands_multitask(
            validation_model,
            x_gpu,
            batch_size=B,
            context_flank=(
                profile_config
                .context_flank
            ),
            target_length=(
                profile_config
                .target_length
            ),
        )

        # GPU -> CPU immediately.
        plus_presence_logit = (
            plus_presence_gpu[
                :,
                0,
                :,
            ]
            .detach()
            .cpu()
            .numpy()
            .astype(
                np.float32,
                copy=True,
            )
        )

        minus_presence_logit_reverse = (
            minus_presence_reverse_gpu[
                :,
                0,
                :,
            ]
            .detach()
            .cpu()
            .numpy()
            .astype(
                np.float32,
                copy=True,
            )
        )

        plus_intensity = (
            plus_intensity_gpu[
                :,
                0,
                :,
            ]
            .detach()
            .cpu()
            .numpy()
            .astype(
                np.float32,
                copy=True,
            )
        )

        minus_intensity_reverse = (
            minus_intensity_reverse_gpu[
                :,
                0,
                :,
            ]
            .detach()
            .cpu()
            .numpy()
            .astype(
                np.float32,
                copy=True,
            )
        )

        del (
            x_gpu,
            plus_presence_gpu,
            minus_presence_reverse_gpu,
            plus_intensity_gpu,
            minus_intensity_reverse_gpu,
        )

        plus_probability = sigmoid_numpy(
            plus_presence_logit
        )

        minus_probability_reverse = sigmoid_numpy(
            minus_presence_logit_reverse
        )

        plus_count = (
            batch.count[
                :,
                0,
                :,
            ]
            .numpy()
            .astype(
                np.float32,
                copy=False,
            )
        )

        minus_count_reverse = (
            batch.count[
                :,
                1,
                :,
            ]
            .flip(-1)
            .numpy()
            .astype(
                np.float32,
                copy=False,
            )
        )

        plus_mask = (
            batch.mask[
                :,
                0,
                :,
            ]
            .numpy()
            .astype(
                bool,
                copy=False,
            )
        )

        minus_mask_reverse = (
            batch.mask[
                :,
                1,
                :,
            ]
            .flip(-1)
            .numpy()
            .astype(
                bool,
                copy=False,
            )
        )

        for local_i in range(B):
            contig = str(
                selected.iloc[
                    local_i
                ]["contig"]
            )

            for (
                probability,
                intensity,
                count,
                mask,
            ) in (
                (
                    plus_probability[
                        local_i
                    ],
                    plus_intensity[
                        local_i
                    ],
                    plus_count[
                        local_i
                    ],
                    plus_mask[
                        local_i
                    ],
                ),
                (
                    minus_probability_reverse[
                        local_i
                    ],
                    minus_intensity_reverse[
                        local_i
                    ],
                    minus_count_reverse[
                        local_i
                    ],
                    minus_mask_reverse[
                        local_i
                    ],
                ),
            ):
                valid_probability = (
                    probability[
                        mask
                    ]
                )

                valid_intensity = (
                    intensity[
                        mask
                    ]
                )

                valid_count = (
                    count[
                        mask
                    ]
                )

                positive = (
                    valid_count > 0
                )

                score_bin = (
                    probability_bin_5dp(
                        valid_probability
                    )
                )

                for segment in (
                    "overall",
                    contig,
                ):
                    state = states[
                        segment
                    ]

                    update_histogram(
                        state[
                            "total_hist"
                        ],
                        score_bin,
                    )

                    if positive.any():
                        update_histogram(
                            state[
                                "positive_hist"
                            ],
                            score_bin[
                                positive
                            ],
                        )

                        state[
                            "positive_presence_raw"
                        ].append(
                            valid_probability[
                                positive
                            ].astype(
                                np.float32,
                                copy=True,
                            )
                        )

                        state[
                            "positive_presence_5dp"
                        ].append(
                            (
                                score_bin[
                                    positive
                                ]
                                / 100_000.0
                            ).astype(
                                np.float32,
                                copy=True,
                            )
                        )

                        state[
                            "positive_intensity"
                        ].append(
                            valid_intensity[
                                positive
                            ].astype(
                                np.float32,
                                copy=True,
                            )
                        )

                        state[
                            "positive_count"
                        ].append(
                            valid_count[
                                positive
                            ].astype(
                                np.float32,
                                copy=True,
                            )
                        )

                negative_index = np.flatnonzero(
                    ~positive
                )

                if len(negative_index):

                    n_random = min(
                        RANDOM_NEG_PER_TILE_STRAND,
                        len(
                            negative_index
                        ),
                    )

                    random_index = rng.choice(
                        negative_index,
                        size=n_random,
                        replace=False,
                    )

                    n_hard = min(
                        HARD_NEG_PER_TILE_STRAND,
                        len(
                            negative_index
                        ),
                    )

                    negative_score = (
                        valid_probability[
                            negative_index
                        ]
                    )

                    if (
                        len(
                            negative_index
                        )
                        <= n_hard
                    ):
                        hard_index = (
                            negative_index
                        )
                    else:
                        local_top = np.argpartition(
                            negative_score,
                            -n_hard,
                        )[
                            -n_hard:
                        ]

                        hard_index = (
                            negative_index[
                                local_top
                            ]
                        )

                    for segment in (
                        "overall",
                        contig,
                    ):
                        states[
                            segment
                        ][
                            "random_negative_score"
                        ].append(
                            valid_probability[
                                random_index
                            ].astype(
                                np.float32,
                                copy=True,
                            )
                        )

                        states[
                            segment
                        ][
                            "hard_negative_score"
                        ].append(
                            valid_probability[
                                hard_index
                            ].astype(
                                np.float32,
                                copy=True,
                            )
                        )

                positions_seen += int(
                    mask.sum()
                )

        del (
            batch,
            selected,
            plus_presence_logit,
            minus_presence_logit_reverse,
            plus_intensity,
            minus_intensity_reverse,
            plus_probability,
            minus_probability_reverse,
            plus_count,
            minus_count_reverse,
            plus_mask,
            minus_mask_reverse,
        )

        if batch_no % 25 == 0:
            gc.collect()

        if (
            batch_no == 1
            or batch_no % PROGRESS_EVERY == 0
            or start + B >= len(
                validation_tiles
            )
        ):
            elapsed = (
                time.perf_counter()
                - started_eval
            )

            print(
                f"tiles={start + B:>6}"
                f"/{len(validation_tiles)} "
                f"positions={positions_seen:>12,} "
                f"positions/s="
                f"{positions_seen / elapsed:,.0f}"
            )

print()
print("FULL HOLDOUT FINISHED")


tiles=     1/47743 positions=         210 positions/s=7,305
tiles=   100/47743 positions=     125,828 positions/s=49,429
tiles=   200/47743 positions=     269,398 positions/s=53,253
tiles=   300/47743 positions=     403,094 positions/s=53,171
tiles=   400/47743 positions=     552,594 positions/s=54,689
tiles=   500/47743 positions=     708,204 positions/s=56,096
tiles=   600/47743 positions=     871,458 positions/s=57,481
tiles=   700/47743 positions=   1,020,090 positions/s=57,697
tiles=   800/47743 positions=   1,185,336 positions/s=58,688
tiles=   900/47743 positions=   1,346,300 positions/s=59,273
tiles=  1000/47743 positions=   1,519,462 positions/s=60,216
tiles=  1100/47743 positions=   1,700,978 positions/s=61,278
tiles=  1200/47743 positions=   1,853,300 positions/s=61,218
tiles=  1300/47743 positions=   2,017,896 positions/s=61,533
tiles=  1400/47743 positions=   2,193,246 positions/s=62,110
tiles=  1500/47743 positions=   2,381,142 positions/s=62,934
tiles=  1600/47743 positi

In [11]:
# ============================================================
# OVERALL + PER-CONTIG METRICS
# ============================================================

metric_rows = []

cohort_cache = {}

for segment, state in states.items():

    positive_raw = np.concatenate(
        state[
            "positive_presence_raw"
        ]
    )

    positive_5dp = np.concatenate(
        state[
            "positive_presence_5dp"
        ]
    )

    positive_intensity = np.concatenate(
        state[
            "positive_intensity"
        ]
    )

    positive_count = np.concatenate(
        state[
            "positive_count"
        ]
    )

    random_negative = np.concatenate(
        state[
            "random_negative_score"
        ]
    )

    hard_negative_candidates = np.concatenate(
        state[
            "hard_negative_score"
        ]
    )

    # Candidate pool contains top negatives from every tile.
    # Keep only its global highest tail.
    hard_keep = min(
        20_000,
        len(
            hard_negative_candidates
        ),
    )

    if (
        len(
            hard_negative_candidates
        )
        > hard_keep
    ):
        hard_idx = np.argpartition(
            hard_negative_candidates,
            -hard_keep,
        )[
            -hard_keep:
        ]

        hard_negative = (
            hard_negative_candidates[
                hard_idx
            ]
        )
    else:
        hard_negative = (
            hard_negative_candidates
        )

    presence_ap_5dp = (
        ap_from_histograms(
            state[
                "positive_hist"
            ],
            state[
                "total_hist"
            ],
        )
    )

    presence_spearman_raw = (
        dense_rank_pearson(
            positive_raw,
            positive_count,
        )
    )

    presence_spearman_5dp = (
        dense_rank_pearson(
            positive_5dp,
            positive_count,
        )
    )

    intensity_spearman = (
        dense_rank_pearson(
            positive_intensity,
            positive_count,
        )
    )

    metric_rows.append({
        "segment": segment,
        "positive_n": int(
            len(
                positive_count
            )
        ),
        "presence_AP_5dp": float(
            presence_ap_5dp
        ),
        "presence_EPIC_Spearman_raw": float(
            presence_spearman_raw
        ),
        "presence_EPIC_Spearman_5dp": float(
            presence_spearman_5dp
        ),
        "intensity_EPIC_Spearman_raw": float(
            intensity_spearman
        ),
    })

    cohort_cache[
        segment
    ] = {
        "positive_score": (
            positive_raw
        ),
        "positive_count": (
            positive_count
        ),
        "random_negative_score": (
            random_negative
        ),
        "hard_negative_score": (
            hard_negative
        ),
    }

metrics = pd.DataFrame(
    metric_rows
).sort_values(
    by="segment"
).reset_index(
    drop=True
)

METRICS_PATH = (
    RUN_DIR
    / "strong_validation_metrics.csv"
)

metrics.to_csv(
    METRICS_PATH,
    index=False,
)

display(metrics)

print("Saved:", METRICS_PATH)


,segment,positive_n,presence_AP_5dp,presence_EPIC_Spearman_raw,presence_EPIC_Spearman_5dp,intensity_EPIC_Spearman_raw
0,NC_064034.1,23529,0.095691,0.216091,0.180568,0.246100
1,NC_064041.1,13984,0.091407,0.247024,0.201083,0.271768
2,NC_064042.1,18692,0.105202,0.238226,0.195032,0.266728
3,overall,56205,0.097630,0.200124,0.148585,0.227598


Saved: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP045_WIDTH512_50K_TWO_PHASE_LR_RF1029\run_001\strong_validation_metrics.csv


## Ошибки: TP, LOW_TP, HARD_FP, TN

Это диагностика вопроса:

> **Что изменилось именно после 128 → 256 channels?**

Для overall:

```text
HIGH_TP = верхние 25% baseline score среди positives
LOW_TP  = нижние 25% baseline score среди positives

HARD_FP = самые высоко оценённые negative candidates
RANDOM_TN = случайная natural-background выборка negatives
```

Смотрим:
- вырос ли AP;
- стабилен ли прирост по contigs;
- изменился ли LOW_TP;
- изменился ли HARD_FP;
- появился ли в 512-D representation новый полезный signal.


In [12]:
# ============================================================
# ERROR-POPULATION STUDY
# ============================================================

overall = cohort_cache[
    "overall"
]

positive_score = overall[
    "positive_score"
]

positive_count = overall[
    "positive_count"
]

random_tn = overall[
    "random_negative_score"
]

hard_fp = overall[
    "hard_negative_score"
]

q25 = float(
    np.quantile(
        positive_score,
        0.25,
    )
)

q75 = float(
    np.quantile(
        positive_score,
        0.75,
    )
)

low_tp = (
    positive_score[
        positive_score <= q25
    ]
)

high_tp = (
    positive_score[
        positive_score >= q75
    ]
)


def describe_scores(
    name,
    values,
):
    values = np.asarray(
        values,
        dtype=np.float64,
    )

    return {
        "cohort": name,
        "n": int(
            len(values)
        ),
        "mean": float(
            values.mean()
        ),
        "p10": float(
            np.quantile(
                values,
                0.10,
            )
        ),
        "p50": float(
            np.quantile(
                values,
                0.50,
            )
        ),
        "p90": float(
            np.quantile(
                values,
                0.90,
            )
        ),
        "p99": float(
            np.quantile(
                values,
                0.99,
            )
        ),
    }


cohort_summary = pd.DataFrame([
    describe_scores(
        "HIGH_TP",
        high_tp,
    ),
    describe_scores(
        "LOW_TP",
        low_tp,
    ),
    describe_scores(
        "ALL_TP",
        positive_score,
    ),
    describe_scores(
        "HARD_FP",
        hard_fp,
    ),
    describe_scores(
        "RANDOM_TN",
        random_tn,
    ),
])

COHORT_PATH = (
    RUN_DIR
    / "error_population_score_summary.csv"
)

cohort_summary.to_csv(
    COHORT_PATH,
    index=False,
)

display(cohort_summary)

print("Positive q25:", q25)
print("Positive q75:", q75)
print("Saved:", COHORT_PATH)


,cohort,n,mean,p10,p50,p90,p99
0,HIGH_TP,14052,0.998413,0.995752,0.999039,0.999931,0.999994
1,LOW_TP,14052,0.363790,0.046659,0.374815,0.662839,0.709836
2,ALL_TP,56205,0.800277,0.286450,0.948561,0.999376,0.999976
3,HARD_FP,20000,0.998309,0.996600,0.998413,0.999803,0.999982
4,RANDOM_TN,190962,0.135465,0.001158,0.025519,0.487429,0.950233


Positive q25: 0.7146850824356079
Positive q75: 0.994308352470398
Saved: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP045_WIDTH512_50K_TWO_PHASE_LR_RF1029\run_001\error_population_score_summary.csv


In [13]:
# ============================================================
# POSITIVE STRENGTH STUDY
# ============================================================

strength_group = np.select(
    [
        positive_count == 1,
        positive_count == 2,
        (
            positive_count >= 3
        )
        & (
            positive_count <= 4
        ),
        positive_count >= 5,
    ],
    [
        "reads_1",
        "reads_2",
        "reads_3_4",
        "reads_5_plus",
    ],
    default="other",
)

strength_rows = []

for group in (
    "reads_1",
    "reads_2",
    "reads_3_4",
    "reads_5_plus",
):
    values = (
        positive_score[
            strength_group == group
        ]
    )

    if len(values) == 0:
        continue

    row = describe_scores(
        group,
        values,
    )

    strength_rows.append(
        row
    )

strength_summary = pd.DataFrame(
    strength_rows
)

STRENGTH_PATH = (
    RUN_DIR
    / "positive_strength_score_summary.csv"
)

strength_summary.to_csv(
    STRENGTH_PATH,
    index=False,
)

display(strength_summary)

print("Saved:", STRENGTH_PATH)


,cohort,n,mean,p10,p50,p90,p99
0,reads_1,29982,0.745592,0.196646,0.892900,0.997207,0.999907
1,reads_2,10496,0.794697,0.276777,0.941441,0.999023,0.999948
2,reads_3_4,6529,0.853175,0.449397,0.974338,0.999528,0.999966
3,reads_5_plus,9198,0.947346,0.866359,0.997137,0.999928,0.999995


Saved: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP045_WIDTH512_50K_TWO_PHASE_LR_RF1029\run_001\positive_strength_score_summary.csv


# OPTIONAL — WIDTH512 representation @50k

Representation diagnostics выполняются только для финального checkpoint.

Ключевой вопрос:

```text
WIDTH256 зрелый representation
vs
WIDTH512 @50k
```

Probe остаётся диагностикой и не заменяет natural-background AP.


In [14]:
# ============================================================
# OPTIONAL FEATURE EXTRACTION
# ============================================================

RUN_FEATURE_PROBE = True

DIAG_TILES_PER_CONTIG = 1000
RANDOM_NEG_FEATURES_PER_TILE_STRAND = 2
HARD_NEG_FEATURES_PER_TILE_STRAND = 2

FEATURES_NPZ = (
    RUN_DIR
    / "width512_50k_feature_probe_samples.npz"
)

if RUN_FEATURE_PROBE:

    diag_parts = []

    for contig, frame in (
        validation_tiles
        .groupby(
            "contig",
            sort=True,
        )
    ):
        n = min(
            DIAG_TILES_PER_CONTIG,
            len(frame),
        )

        positions = np.linspace(
            0,
            len(frame) - 1,
            n,
            dtype=int,
        )

        positions = np.unique(
            positions
        )

        diag_parts.append(
            frame.iloc[
                positions
            ]
        )

    diag_tiles = (
        pd.concat(
            diag_parts,
            axis=0,
        )
        .reset_index(
            drop=True
        )
    )

    print(
        "Diagnostic tiles:",
        len(diag_tiles),
    )

    rng_probe = np.random.default_rng(
        20260925
    )

    tp_X = []
    tp_score = []
    tp_count = []
    tp_contig = []

    tn_X = []
    tn_score = []
    tn_contig = []

    hard_X = []
    hard_score = []
    hard_contig = []

    validation_model.eval()

    started_probe = time.perf_counter()

    with torch.no_grad():

        for row_i in range(
            len(diag_tiles)
        ):
            selected = (
                diag_tiles
                .iloc[
                    row_i:
                    row_i + 1
                ]
            )

            batch = (
                validation_generator
                .make_batch(
                    selected
                )
            )

            contig = str(
                selected.iloc[
                    0
                ]["contig"]
            )

            x_gpu = (
                batch
                .x_both_strands
                .to(device)
            )

            feature_gpu = (
                validation_model
                .backbone(
                    x_gpu
                )
            )

            output_gpu = (
                validation_model
                .profile_head(
                    feature_gpu
                )
            )

            # CPU first.
            feature = (
                feature_gpu
                .detach()
                .cpu()
                .numpy()
                .astype(
                    np.float32,
                    copy=True,
                )
            )

            output = (
                output_gpu
                .detach()
                .cpu()
                .numpy()
                .astype(
                    np.float32,
                    copy=True,
                )
            )

            del (
                x_gpu,
                feature_gpu,
                output_gpu,
            )

            c = (
                profile_config
                .context_flank
            )

            L = (
                profile_config
                .target_length
            )

            feature = (
                feature[
                    :,
                    :,
                    c:
                    c + L
                ]
            )

            output = (
                output[
                    :,
                    :,
                    c:
                    c + L
                ]
            )

            # B=1:
            # index 0 = plus orientation
            # index 1 = reverse-complement orientation.
            plus_count = (
                batch.count[
                    0,
                    0,
                    :,
                ]
                .numpy()
                .astype(
                    np.float32,
                    copy=False,
                )
            )

            minus_count_reverse = (
                batch.count[
                    0,
                    1,
                    :,
                ]
                .flip(-1)
                .numpy()
                .astype(
                    np.float32,
                    copy=False,
                )
            )

            plus_mask = (
                batch.mask[
                    0,
                    0,
                    :,
                ]
                .numpy()
                .astype(
                    bool,
                    copy=False,
                )
            )

            minus_mask_reverse = (
                batch.mask[
                    0,
                    1,
                    :,
                ]
                .flip(-1)
                .numpy()
                .astype(
                    bool,
                    copy=False,
                )
            )

            for orient_i, (
                count,
                mask,
            ) in enumerate(
                (
                    (
                        plus_count,
                        plus_mask,
                    ),
                    (
                        minus_count_reverse,
                        minus_mask_reverse,
                    ),
                )
            ):
                feat = (
                    feature[
                        orient_i
                    ]
                    .T
                )

                probability = (
                    sigmoid_numpy(
                        output[
                            orient_i,
                            0,
                            :,
                        ]
                    )
                )

                valid_index = np.flatnonzero(
                    mask
                )

                pos_index = valid_index[
                    count[
                        valid_index
                    ] > 0
                ]

                neg_index = valid_index[
                    count[
                        valid_index
                    ] == 0
                ]

                if len(pos_index):
                    tp_X.append(
                        feat[
                            pos_index
                        ].copy()
                    )

                    tp_score.append(
                        probability[
                            pos_index
                        ].copy()
                    )

                    tp_count.append(
                        count[
                            pos_index
                        ].copy()
                    )

                    tp_contig.extend(
                        [
                            contig
                        ]
                        * len(
                            pos_index
                        )
                    )

                if len(neg_index):
                    n_random = min(
                        RANDOM_NEG_FEATURES_PER_TILE_STRAND,
                        len(
                            neg_index
                        ),
                    )

                    random_index = rng_probe.choice(
                        neg_index,
                        size=n_random,
                        replace=False,
                    )

                    n_hard = min(
                        HARD_NEG_FEATURES_PER_TILE_STRAND,
                        len(
                            neg_index
                        ),
                    )

                    neg_probability = (
                        probability[
                            neg_index
                        ]
                    )

                    if len(
                        neg_index
                    ) <= n_hard:
                        hard_index = (
                            neg_index
                        )
                    else:
                        local_top = np.argpartition(
                            neg_probability,
                            -n_hard,
                        )[
                            -n_hard:
                        ]

                        hard_index = (
                            neg_index[
                                local_top
                            ]
                        )

                    tn_X.append(
                        feat[
                            random_index
                        ].copy()
                    )

                    tn_score.append(
                        probability[
                            random_index
                        ].copy()
                    )

                    tn_contig.extend(
                        [
                            contig
                        ]
                        * len(
                            random_index
                        )
                    )

                    hard_X.append(
                        feat[
                            hard_index
                        ].copy()
                    )

                    hard_score.append(
                        probability[
                            hard_index
                        ].copy()
                    )

                    hard_contig.extend(
                        [
                            contig
                        ]
                        * len(
                            hard_index
                        )
                    )

            del (
                batch,
                selected,
                feature,
                output,
            )

            if (
                row_i == 0
                or (
                    row_i + 1
                ) % 250 == 0
                or row_i + 1
                == len(
                    diag_tiles
                )
            ):
                elapsed = (
                    time.perf_counter()
                    - started_probe
                )

                print(
                    f"probe tiles="
                    f"{row_i + 1:>5}"
                    f"/{len(diag_tiles)} "
                    f"elapsed="
                    f"{elapsed / 60:.1f} min"
                )

            if (
                row_i + 1
            ) % 25 == 0:
                gc.collect()

    tp_X = np.concatenate(
        tp_X,
        axis=0,
    )

    tp_score = np.concatenate(
        tp_score
    )

    tp_count = np.concatenate(
        tp_count
    )

    tp_contig = np.asarray(
        tp_contig
    )

    tn_X = np.concatenate(
        tn_X,
        axis=0,
    )

    tn_score = np.concatenate(
        tn_score
    )

    tn_contig = np.asarray(
        tn_contig
    )

    hard_X = np.concatenate(
        hard_X,
        axis=0,
    )

    hard_score = np.concatenate(
        hard_score
    )

    hard_contig = np.asarray(
        hard_contig
    )

    np.savez_compressed(
        FEATURES_NPZ,
        tp_X=tp_X,
        tp_score=tp_score,
        tp_count=tp_count,
        tp_contig=tp_contig,
        random_tn_X=tn_X,
        random_tn_score=tn_score,
        random_tn_contig=tn_contig,
        hard_fp_X=hard_X,
        hard_fp_score=hard_score,
        hard_fp_contig=hard_contig,
    )

    print()
    print("TP:", tp_X.shape)
    print("Random TN:", tn_X.shape)
    print("Hard FP:", hard_X.shape)
    print("Saved:", FEATURES_NPZ)

else:
    print("Feature probe disabled.")


Diagnostic tiles: 3000
probe tiles=    1/3000 elapsed=0.0 min
probe tiles=  250/3000 elapsed=0.1 min
probe tiles=  500/3000 elapsed=0.2 min
probe tiles=  750/3000 elapsed=0.3 min
probe tiles= 1000/3000 elapsed=0.5 min
probe tiles= 1250/3000 elapsed=0.6 min
probe tiles= 1500/3000 elapsed=0.7 min
probe tiles= 1750/3000 elapsed=0.8 min
probe tiles= 2000/3000 elapsed=0.9 min
probe tiles= 2250/3000 elapsed=1.0 min
probe tiles= 2500/3000 elapsed=1.1 min
probe tiles= 2750/3000 elapsed=1.3 min
probe tiles= 3000/3000 elapsed=1.4 min

TP: (3483, 512)
Random TN: (11998, 512)
Hard FP: (11998, 512)
Saved: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP045_WIDTH512_50K_TWO_PHASE_LR_RF1029\run_001\width512_50k_feature_probe_samples.npz


In [15]:
# ============================================================
# OPTIONAL CROSS-CONTIG LINEAR PROBE
# ============================================================

if RUN_FEATURE_PROBE:

    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import roc_auc_score
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler

    data = np.load(
        FEATURES_NPZ,
        allow_pickle=False,
    )

    tp_X = data[
        "tp_X"
    ]

    tp_score = data[
        "tp_score"
    ]

    tp_contig = data[
        "tp_contig"
    ]

    random_tn_X = data[
        "random_tn_X"
    ]

    random_tn_score = data[
        "random_tn_score"
    ]

    random_tn_contig = data[
        "random_tn_contig"
    ]

    hard_fp_X = data[
        "hard_fp_X"
    ]

    hard_fp_score = data[
        "hard_fp_score"
    ]

    hard_fp_contig = data[
        "hard_fp_contig"
    ]


    def cross_contig_probe(
        positive_X,
        positive_score,
        positive_contig,
        negative_X,
        negative_score,
        negative_contig,
        task_name,
    ):
        rows = []

        contigs = sorted(
            set(
                positive_contig.tolist()
            )
            & set(
                negative_contig.tolist()
            )
        )

        for heldout in contigs:

            pos_train = (
                positive_contig
                != heldout
            )

            neg_train = (
                negative_contig
                != heldout
            )

            pos_test = (
                positive_contig
                == heldout
            )

            neg_test = (
                negative_contig
                == heldout
            )

            X_train = np.concatenate(
                (
                    positive_X[
                        pos_train
                    ],
                    negative_X[
                        neg_train
                    ],
                ),
                axis=0,
            )

            y_train = np.concatenate(
                (
                    np.ones(
                        pos_train.sum(),
                        dtype=np.int8,
                    ),
                    np.zeros(
                        neg_train.sum(),
                        dtype=np.int8,
                    ),
                )
            )

            X_test = np.concatenate(
                (
                    positive_X[
                        pos_test
                    ],
                    negative_X[
                        neg_test
                    ],
                ),
                axis=0,
            )

            y_test = np.concatenate(
                (
                    np.ones(
                        pos_test.sum(),
                        dtype=np.int8,
                    ),
                    np.zeros(
                        neg_test.sum(),
                        dtype=np.int8,
                    ),
                )
            )

            head_score = np.concatenate(
                (
                    positive_score[
                        pos_test
                    ],
                    negative_score[
                        neg_test
                    ],
                )
            )

            probe = make_pipeline(
                StandardScaler(),
                LogisticRegression(
                    max_iter=1000,
                    class_weight="balanced",
                    solver="lbfgs",
                ),
            )

            probe.fit(
                X_train,
                y_train,
            )

            probe_score = (
                probe.predict_proba(
                    X_test
                )[
                    :,
                    1
                ]
            )

            rows.append({
                "task": task_name,
                "heldout_contig": heldout,
                "n_test": int(
                    len(
                        y_test
                    )
                ),
                "presence_head_ROC_AUC": float(
                    roc_auc_score(
                        y_test,
                        head_score,
                    )
                ),
                "linear_probe_ROC_AUC": float(
                    roc_auc_score(
                        y_test,
                        probe_score,
                    )
                ),
            })

        return pd.DataFrame(
            rows
        )


    probe_random = cross_contig_probe(
        tp_X,
        tp_score,
        tp_contig,
        random_tn_X,
        random_tn_score,
        random_tn_contig,
        "TP_vs_RANDOM_TN",
    )

    probe_hard = cross_contig_probe(
        tp_X,
        tp_score,
        tp_contig,
        hard_fp_X,
        hard_fp_score,
        hard_fp_contig,
        "TP_vs_HARD_FP",
    )

    probe_results = pd.concat(
        (
            probe_random,
            probe_hard,
        ),
        ignore_index=True,
    )

    PROBE_PATH = (
        RUN_DIR
        / "width512_50k_cross_contig_feature_probe.csv"
    )

    probe_results.to_csv(
        PROBE_PATH,
        index=False,
    )

    display(probe_results)

    display(
        probe_results
        .groupby(
            "task",
            as_index=False,
        )[
            [
                "presence_head_ROC_AUC",
                "linear_probe_ROC_AUC",
            ]
        ]
        .mean()
    )

    print("Saved:", PROBE_PATH)

else:
    print("Feature probe disabled.")


,task,heldout_contig,n_test,presence_head_ROC_AUC,linear_probe_ROC_AUC
0,TP_vs_RANDOM_TN,NC_064034.1,5115,0.943328,0.929924
1,TP_vs_RANDOM_TN,NC_064041.1,5035,0.957427,0.947920
2,TP_vs_RANDOM_TN,NC_064042.1,5331,0.942813,0.925391
3,TP_vs_HARD_FP,NC_064034.1,5115,0.471151,0.815751
4,TP_vs_HARD_FP,NC_064041.1,5035,0.567637,0.815713
5,TP_vs_HARD_FP,NC_064042.1,5331,0.449693,0.818072


,task,presence_head_ROC_AUC,linear_probe_ROC_AUC
0,TP_vs_HARD_FP,0.496161,0.816512
1,TP_vs_RANDOM_TN,0.947856,0.934412


Saved: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP045_WIDTH512_50K_TWO_PHASE_LR_RF1029\run_001\width512_50k_cross_contig_feature_probe.csv


# LOW_TP vs score-matched natural TN — residual-information probe

Это ключевой representation test после WIDTH512.

Мы **не обучаем residual correction** и не меняем CNN.

Для каждого held-out validation contig:

```text
train:
    LOW_TP из остальных validation contigs
    vs score-matched random TN

test:
    LOW_TP held-out contig
    vs score-matched random TN held-out contig
```

LOW_TP = нижний квартиль TP по baseline presence score.

TN подбираются из тех же score ranges. Поэтому:

```text
baseline-score AUC ≈ 0.5
linear 512-D probe AUC > 0.5
```

означает, что representation содержит дополнительную информацию,
которой scalar presence score не использует.

Если доступен EXP036 `width64_feature_probe_samples.npz`,
тем же кодом считаем WIDTH64 для прямого сравнения.


In [16]:
# ============================================================
# LOW_TP vs SCORE-MATCHED RANDOM TN PROBE
# ============================================================

LOWTP_PROBE_PATH = (
    RUN_DIR
    / "width512_50k_lowtp_scorematched_probe.csv"
)

WIDTH_COMPARISON_PATH = (
    RUN_DIR
    / "width64_vs_width128_vs_width512_50k_lowtp_scorematched_probe.csv"
)


def score_matched_negative_indices(
    positive_scores,
    negative_scores,
    *,
    seed,
    n_bins=10,
):
    rng = np.random.default_rng(seed)

    positive_scores = np.asarray(
        positive_scores,
        dtype=np.float64,
    )

    negative_scores = np.asarray(
        negative_scores,
        dtype=np.float64,
    )

    edges = np.unique(
        np.quantile(
            positive_scores,
            np.linspace(
                0.0,
                1.0,
                n_bins + 1,
            ),
        )
    )

    if len(edges) < 3:
        lo = float(
            positive_scores.min()
        )
        hi = float(
            positive_scores.max()
        )

        if hi <= lo:
            hi = lo + 1e-6

        edges = np.linspace(
            lo,
            hi,
            3,
        )

    chosen_parts = []

    for bin_i in range(
        len(edges) - 1
    ):
        left = edges[bin_i]
        right = edges[bin_i + 1]

        if bin_i == len(edges) - 2:
            pos_mask = (
                (positive_scores >= left)
                & (positive_scores <= right)
            )
            neg_mask = (
                (negative_scores >= left)
                & (negative_scores <= right)
            )
        else:
            pos_mask = (
                (positive_scores >= left)
                & (positive_scores < right)
            )
            neg_mask = (
                (negative_scores >= left)
                & (negative_scores < right)
            )

        n_pos = int(
            pos_mask.sum()
        )

        if n_pos == 0:
            continue

        candidates = np.flatnonzero(
            neg_mask
        )

        if len(candidates) == 0:
            center = 0.5 * (
                left + right
            )

            distance = np.abs(
                negative_scores
                - center
            )

            fallback_n = min(
                max(
                    n_pos,
                    16,
                ),
                len(
                    negative_scores
                ),
            )

            candidates = np.argpartition(
                distance,
                fallback_n - 1,
            )[
                :fallback_n
            ]

        selected = rng.choice(
            candidates,
            size=n_pos,
            replace=(
                n_pos
                > len(candidates)
            ),
        )

        chosen_parts.append(
            selected
        )

    if not chosen_parts:
        raise RuntimeError(
            "No score-matched TN candidates."
        )

    return np.concatenate(
        chosen_parts
    )


def lowtp_scorematched_probe_from_npz(
    npz_path,
    *,
    representation_name,
):
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import roc_auc_score
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler

    data = np.load(
        npz_path,
        allow_pickle=False,
    )

    X_pos = data["tp_X"]
    score_pos = data["tp_score"]
    contig_pos = data["tp_contig"]

    X_neg = data["random_tn_X"]
    score_neg = data["random_tn_score"]
    contig_neg = data["random_tn_contig"]

    rows = []

    contigs = sorted(
        set(
            contig_pos.tolist()
        )
        & set(
            contig_neg.tolist()
        )
    )

    for fold_i, heldout in enumerate(
        contigs,
        1,
    ):
        train_pos_mask = (
            contig_pos != heldout
        )
        test_pos_mask = (
            contig_pos == heldout
        )

        train_neg_mask = (
            contig_neg != heldout
        )
        test_neg_mask = (
            contig_neg == heldout
        )

        # LOW_TP is defined within train/test separately by baseline score.
        train_cutoff = float(
            np.quantile(
                score_pos[
                    train_pos_mask
                ],
                0.25,
            )
        )

        test_cutoff = float(
            np.quantile(
                score_pos[
                    test_pos_mask
                ],
                0.25,
            )
        )

        train_low_mask = (
            train_pos_mask
            & (
                score_pos
                <= train_cutoff
            )
        )

        test_low_mask = (
            test_pos_mask
            & (
                score_pos
                <= test_cutoff
            )
        )

        train_neg_idx_all = np.flatnonzero(
            train_neg_mask
        )

        test_neg_idx_all = np.flatnonzero(
            test_neg_mask
        )

        train_match_local = (
            score_matched_negative_indices(
                score_pos[
                    train_low_mask
                ],
                score_neg[
                    train_neg_mask
                ],
                seed=20260925 + fold_i,
            )
        )

        test_match_local = (
            score_matched_negative_indices(
                score_pos[
                    test_low_mask
                ],
                score_neg[
                    test_neg_mask
                ],
                seed=20261925 + fold_i,
            )
        )

        train_neg_idx = (
            train_neg_idx_all[
                train_match_local
            ]
        )

        test_neg_idx = (
            test_neg_idx_all[
                test_match_local
            ]
        )

        X_train = np.concatenate(
            (
                X_pos[
                    train_low_mask
                ],
                X_neg[
                    train_neg_idx
                ],
            ),
            axis=0,
        )

        y_train = np.concatenate(
            (
                np.ones(
                    int(
                        train_low_mask.sum()
                    ),
                    dtype=np.int8,
                ),
                np.zeros(
                    len(
                        train_neg_idx
                    ),
                    dtype=np.int8,
                ),
            )
        )

        X_test = np.concatenate(
            (
                X_pos[
                    test_low_mask
                ],
                X_neg[
                    test_neg_idx
                ],
            ),
            axis=0,
        )

        y_test = np.concatenate(
            (
                np.ones(
                    int(
                        test_low_mask.sum()
                    ),
                    dtype=np.int8,
                ),
                np.zeros(
                    len(
                        test_neg_idx
                    ),
                    dtype=np.int8,
                ),
            )
        )

        baseline_score = np.concatenate(
            (
                score_pos[
                    test_low_mask
                ],
                score_neg[
                    test_neg_idx
                ],
            )
        )

        probe = make_pipeline(
            StandardScaler(),
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                solver="lbfgs",
            ),
        )

        probe.fit(
            X_train,
            y_train,
        )

        probe_score = (
            probe
            .predict_proba(
                X_test
            )[
                :,
                1
            ]
        )

        rows.append({
            "representation": representation_name,
            "heldout_contig": heldout,
            "n_low_tp_test": int(
                test_low_mask.sum()
            ),
            "n_scorematched_tn_test": int(
                len(
                    test_neg_idx
                )
            ),
            "baseline_score_ROC_AUC": float(
                roc_auc_score(
                    y_test,
                    baseline_score,
                )
            ),
            "linear_probe_ROC_AUC": float(
                roc_auc_score(
                    y_test,
                    probe_score,
                )
            ),
        })

    return pd.DataFrame(
        rows
    )




if RUN_FEATURE_PROBE:

    width512_probe = (
        lowtp_scorematched_probe_from_npz(
            FEATURES_NPZ,
            representation_name="WIDTH512_50K",
        )
    )

    width512_probe.to_csv(
        LOWTP_PROBE_PATH,
        index=False,
    )

    comparison_parts = []

    exp044_features = (
        PROJECT_ROOT
        / "artifacts"
        / "experiments"
        / "CNN_EXP044_WIDTH256_25K_TO_50K_COSINE_LR"
        / "run_001"
        / "width256_cosine_50k_feature_probe_samples.npz"
    )

    if exp044_features.is_file():
        comparison_parts.append(
            lowtp_scorematched_probe_from_npz(
                exp044_features,
                representation_name="WIDTH256_COSINE_50K",
            )
        )
    else:
        print(
            "WIDTH256 mature feature NPZ not found:",
            exp044_features,
        )

    comparison_parts.append(
        width512_probe
    )

    width_compare = pd.concat(
        comparison_parts,
        ignore_index=True,
    )

    width_compare.to_csv(
        WIDTH_COMPARISON_PATH,
        index=False,
    )

    print("LOW_TP representation comparison:")
    display(width_compare)

    print()
    print("Mean probe AUC:")
    display(
        width_compare
        .groupby(
            "representation",
            as_index=False,
        )[
            [
                "baseline_score_ROC_AUC",
                "linear_probe_ROC_AUC",
            ]
        ]
        .mean()
    )

    print("Saved:", LOWTP_PROBE_PATH)
    print("Saved:", WIDTH_COMPARISON_PATH)

else:
    print(
        "RUN_FEATURE_PROBE=False; "
        "LOW_TP probe skipped."
    )


LOW_TP representation comparison:


,representation,heldout_contig,n_low_tp_test,n_scorematched_tn_test,baseline_score_ROC_AUC,linear_probe_ROC_AUC
0,WIDTH256_COSINE_50K,NC_064034.1,279,279,0.501869,0.548927
1,WIDTH256_COSINE_50K,NC_064041.1,259,259,0.502020,0.538752
2,WIDTH256_COSINE_50K,NC_064042.1,334,334,0.501721,0.508390
3,WIDTH512_50K,NC_064034.1,279,279,0.504528,0.520022
4,WIDTH512_50K,NC_064041.1,259,259,0.503868,0.510845
5,WIDTH512_50K,NC_064042.1,334,334,0.504787,0.558092



Mean probe AUC:


,representation,baseline_score_ROC_AUC,linear_probe_ROC_AUC
0,WIDTH256_COSINE_50K,0.501870,0.532023
1,WIDTH512_50K,0.504395,0.529653


Saved: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP045_WIDTH512_50K_TWO_PHASE_LR_RF1029\run_001\width512_50k_lowtp_scorematched_probe.csv
Saved: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP045_WIDTH512_50K_TWO_PHASE_LR_RF1029\run_001\width64_vs_width128_vs_width512_50k_lowtp_scorematched_probe.csv


# Сравнение с зрелым WIDTH256

Главный архитектурный reference:

```text
EXP044 WIDTH256 @50k
LR:
0→25k = 1e-3
25k→50k = cosine 3e-4→3e-5
```

Candidate:

```text
EXP045 WIDTH512 @50k
тот же двухфазный LR schedule
тот же RF1029
```

Так сравнение WIDTH256 → WIDTH512 максимально близко к чистому capacity scaling.


In [17]:
# ============================================================
# REFERENCE DISPLAY — MATURE WIDTH256
# ============================================================

EXP044_RUN = (
    PROJECT_ROOT
    / "artifacts"
    / "experiments"
    / "CNN_EXP044_WIDTH256_25K_TO_50K_COSINE_LR"
    / "run_001"
)

REFERENCE_METRICS_PATH = (
    EXP044_RUN
    / "strong_validation_metrics.csv"
)

print("WIDTH512 @50k:")
display(metrics)
print()

if REFERENCE_METRICS_PATH.is_file():

    print("WIDTH256 cosine @50k — EXP044:")

    reference_metrics = pd.read_csv(
        REFERENCE_METRICS_PATH
    )

    display(reference_metrics)

    comparison = (
        reference_metrics[
            [
                "segment",
                "presence_AP_5dp",
                "presence_EPIC_Spearman_raw",
                "intensity_EPIC_Spearman_raw",
            ]
        ]
        .merge(
            metrics[
                [
                    "segment",
                    "presence_AP_5dp",
                    "presence_EPIC_Spearman_raw",
                    "intensity_EPIC_Spearman_raw",
                ]
            ],
            on="segment",
            suffixes=(
                "_width256",
                "_width512",
            ),
        )
    )

    comparison[
        "delta_AP_512_minus_256"
    ] = (
        comparison[
            "presence_AP_5dp_width512"
        ]
        - comparison[
            "presence_AP_5dp_width256"
        ]
    )

    display(comparison)

    WIDTH_REFERENCE_COMPARISON_PATH = (
        RUN_DIR
        / "width256_vs_width512_50k_metrics.csv"
    )

    comparison.to_csv(
        WIDTH_REFERENCE_COMPARISON_PATH,
        index=False,
    )

    print(
        "Saved:",
        WIDTH_REFERENCE_COMPARISON_PATH,
    )

else:

    WIDTH_REFERENCE_COMPARISON_PATH = None

    print(
        "EXP044 mature WIDTH256 metrics not found:",
        REFERENCE_METRICS_PATH,
    )


WIDTH512 @50k:


,segment,positive_n,presence_AP_5dp,presence_EPIC_Spearman_raw,presence_EPIC_Spearman_5dp,intensity_EPIC_Spearman_raw
0,NC_064034.1,23529,0.095691,0.216091,0.180568,0.246100
1,NC_064041.1,13984,0.091407,0.247024,0.201083,0.271768
2,NC_064042.1,18692,0.105202,0.238226,0.195032,0.266728
3,overall,56205,0.097630,0.200124,0.148585,0.227598



WIDTH256 cosine @50k — EXP044:


,segment,positive_n,presence_AP_5dp,presence_EPIC_Spearman_raw,presence_EPIC_Spearman_5dp,intensity_EPIC_Spearman_raw
0,NC_064034.1,23529,0.092443,0.212595,0.177833,0.244876
1,NC_064041.1,13984,0.088142,0.241665,0.200462,0.270932
2,NC_064042.1,18692,0.098943,0.233618,0.191937,0.267621
3,overall,56205,0.093362,0.195897,0.146555,0.226628


,segment,presence_AP_5dp_width256,presence_EPIC_Spearman_raw_width256,intensity_EPIC_Spearman_raw_width256,presence_AP_5dp_width512,presence_EPIC_Spearman_raw_width512,intensity_EPIC_Spearman_raw_width512,delta_AP_512_minus_256
0,NC_064034.1,0.092443,0.212595,0.244876,0.095691,0.216091,0.246100,0.003248
1,NC_064041.1,0.088142,0.241665,0.270932,0.091407,0.247024,0.271768,0.003266
2,NC_064042.1,0.098943,0.233618,0.267621,0.105202,0.238226,0.266728,0.006259
3,overall,0.093362,0.195897,0.226628,0.097630,0.200124,0.227598,0.004268


Saved: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP045_WIDTH512_50K_TWO_PHASE_LR_RF1029\run_001\width256_vs_width512_50k_metrics.csv


In [18]:
# ============================================================
# SAVE COMPACT RESULT JSON
# ============================================================

RESULT_JSON = (
    RUN_DIR
    / "EXP045_WIDTH512_50K_RESULT.json"
)

overall_metrics = (
    metrics
    .loc[
        metrics["segment"] == "overall"
    ]
    .iloc[0]
    .to_dict()
)

history_df = pd.read_csv(
    HISTORY_PATH
)

training_hours = float(
    history_df[
        "elapsed_seconds"
    ].max()
    / 3600.0
)

result_payload = {
    "experiment": {
        "id": EXPERIMENT_ID,
        "hypothesis": (
            "Doubling width 256→512 improves natural-background "
            "ranking when both models receive a mature training schedule."
        ),
        "channels": 512,
        "reference_channels": 256,
        "trainable_parameters": 14_708_738,
        "theoretical_rf": 1029,
        "steps": STEPS,
        "batch_size": BATCH_SIZE,
        "phase1": {
            "steps": "1..25000",
            "learning_rate": PHASE1_LR,
        },
        "phase2": {
            "steps": "25001..50000",
            "scheduler": "CosineAnnealingLR",
            "start_lr": PHASE2_START_LR,
            "end_lr": PHASE2_END_LR,
        },
        "only_intended_architecture_change_vs_reference": (
            "backbone width 256→512"
        ),
        "training_hours": training_hours,
    },

    "overall": {
        key: (
            int(value)
            if key == "positive_n"
            else float(value)
        )
        for key, value in overall_metrics.items()
        if key != "segment"
    },

    "reference": {
        "experiment": "CNN-EXP-044",
        "channels": 256,
        "steps": 50_000,
        "schedule": (
            "1e-3 to 25k; "
            "cosine 3e-4→3e-5 to 50k"
        ),
    },

    "files": {
        "checkpoint": str(FINAL_CHECKPOINT),
        "history": str(HISTORY_PATH),
        "strong_validation_metrics": str(METRICS_PATH),
        "width_reference_comparison": (
            str(
                WIDTH_REFERENCE_COMPARISON_PATH
            )
            if WIDTH_REFERENCE_COMPARISON_PATH
            is not None
            else None
        ),
        "error_population_summary": str(COHORT_PATH),
        "positive_strength_summary": str(STRENGTH_PATH),
        "feature_probe_samples": (
            str(FEATURES_NPZ)
            if RUN_FEATURE_PROBE
            else None
        ),
        "feature_probe": (
            str(PROBE_PATH)
            if RUN_FEATURE_PROBE
            else None
        ),
        "lowtp_scorematched_probe": (
            str(LOWTP_PROBE_PATH)
            if RUN_FEATURE_PROBE
            else None
        ),
        "width_representation_comparison": (
            str(WIDTH_COMPARISON_PATH)
            if RUN_FEATURE_PROBE
            else None
        ),
    },

    "decision_questions": [
        "Does WIDTH512 beat mature WIDTH256 on full-holdout Presence AP?",
        "Is any AP gain consistent across all validation contigs?",
        "Does Presence Spearman improve?",
        "Does Intensity Spearman improve?",
        "Does WIDTH512 representation improve LOW_TP diagnostics?",
        "Is width still scaling, or has capacity saturated?",
    ],
}

RESULT_JSON.write_text(
    json.dumps(
        result_payload,
        ensure_ascii=False,
        indent=2,
        allow_nan=False,
    ),
    encoding="utf-8",
)

print("=" * 72)
print("EXP045 COMPLETE")
print("=" * 72)
print()
print(
    "Training hours:",
    round(training_hours, 3),
)
print()
print("Send me:")
print(RESULT_JSON)

if RUN_FEATURE_PROBE:
    print(PROBE_PATH)
    print(LOWTP_PROBE_PATH)
    print(WIDTH_COMPARISON_PATH)


EXP045 COMPLETE

Training hours: 3.165

Send me:
D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP045_WIDTH512_50K_TWO_PHASE_LR_RF1029\run_001\EXP045_WIDTH512_50K_RESULT.json
D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP045_WIDTH512_50K_TWO_PHASE_LR_RF1029\run_001\width512_50k_cross_contig_feature_probe.csv
D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP045_WIDTH512_50K_TWO_PHASE_LR_RF1029\run_001\width512_50k_lowtp_scorematched_probe.csv
D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP045_WIDTH512_50K_TWO_PHASE_LR_RF1029\run_001\width64_vs_width128_vs_width512_50k_lowtp_scorematched_probe.csv


In [19]:
# ============================================================
# OPTIONAL: REGISTER EXPERIMENT AFTER REVIEW
# ============================================================

SAVE_EXPERIMENT = False

FINAL_DECISION = "review"
FINAL_NOTES = (
    "WIDTH512 long-run capacity test. 50k steps. "
    "Two-phase LR: 1e-3 to 25k, then cosine 3e-4→3e-5."
)

REPORT_METRICS_PATH = (
    RUN_DIR
    / "validation_metrics_report.csv"
)

report_metrics = (
    metrics[
        [
            "segment",
            "presence_AP_5dp",
            "presence_EPIC_Spearman_raw",
        ]
    ]
    .rename(
        columns={
            "presence_AP_5dp": "average_precision",
            "presence_EPIC_Spearman_raw": "epic_spearman",
        }
    )
)

report_metrics.to_csv(
    REPORT_METRICS_PATH,
    index=False,
)

EXP044_RUN = (
    PROJECT_ROOT
    / "artifacts"
    / "experiments"
    / "CNN_EXP044_WIDTH256_25K_TO_50K_COSINE_LR"
    / "run_001"
)

REFERENCE_REPORT_PATH = (
    EXP044_RUN
    / "validation_metrics_report.csv"
)

CARD_PATH = (
    PROJECT_ROOT
    / "cnn_exp"
    / f"{EXPERIMENT_ID}.md"
)

CARD_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

if not CARD_PATH.exists():

    card_text = (
        "---\n"
        f"id: {EXPERIMENT_ID}\n"
        "type: experiment\n"
        "experiment_type: cnn-candidate\n"
        "status: completed\n"
        "decision: review\n"
        "date: 2026-09-25\n"
        "seed: 42\n"
        "split_version: contig_holdout_v1\n"
        "primary_metric: average_precision\n"
        "secondary_metric: epic_spearman\n"
        "reference_experiment: CNN-EXP-044\n"
        "tags:\n"
        "  - ml/experiment\n"
        "  - ml/cnn\n"
        "  - ml/capacity\n"
        "  - ml/overnight\n"
        "---\n\n"
        f"# {EXPERIMENT_ID} — WIDTH512 @50k RF1029\n\n"
        "← [[cnn_exp/_index.md|Реестр CNN]]\n\n"
        "## Pre-registration\n\n"
        "- **Гипотеза:** WIDTH512 улучшит ranking относительно зрелого WIDTH256.\n"
        "- **Architecture change:** channels 256 → 512.\n"
        "- **RF:** 1029, unchanged.\n"
        "- **Training:** 50k steps.\n"
        "- **LR:** 1e-3 through 25k; then cosine 3e-4→3e-5 through 50k.\n"
        "- **Batch:** 4.\n"
        "- **Validation:** one final full-holdout strong validation only.\n"
        "- **Reference:** CNN-EXP-044.\n\n"
        "## Автоматический отчёт\n\n"
        "<!-- auto:cnn-experiment-report:start -->\n\n"
        "<!-- auto:cnn-experiment-report:end -->\n\n"
        "## Ручные заметки\n\n"
        "-\n"
    )

    CARD_PATH.write_text(
        card_text,
        encoding="utf-8",
    )

print("Reporter metrics:", REPORT_METRICS_PATH)
print("Reference reporter metrics:", REFERENCE_REPORT_PATH)
print("Card:", CARD_PATH)

if SAVE_EXPERIMENT:

    from ml_project.cnn_experiment_report import (
        finalize_cnn_experiment,
    )

    assert REFERENCE_REPORT_PATH.is_file(), (
        f"Reference reporter metrics missing: "
        f"{REFERENCE_REPORT_PATH}"
    )

    finalization = finalize_cnn_experiment(
        project_root=PROJECT_ROOT,
        experiment_id=EXPERIMENT_ID,
        title="WIDTH512 @50k RF1029",
        hypothesis=(
            "WIDTH512 improves natural-background ranking "
            "relative to mature WIDTH256 with matched training schedule"
        ),
        notebook_path=(
            "notebooks/experiments/CNN_experiments/"
            "CNN_EXP045_WIDTH512_50K_TWO_PHASE_LR_RF1029.ipynb"
        ),
        artifact_dir=RUN_DIR,
        training_history=HISTORY_PATH,
        candidate_metrics=REPORT_METRICS_PATH,
        reference_experiment="CNN-EXP-044",
        reference_metrics=REFERENCE_REPORT_PATH,
        decision=FINAL_DECISION,
        notes=FINAL_NOTES,
    )

    print("Obsidian:", finalization.card_path)
    print("Summary:", finalization.summary_path)
    print("Plots:", finalization.plot_dir)

else:

    print(
        "SAVE_EXPERIMENT=False. "
        "Сначала пришли EXP045 result JSON и metrics."
    )


Reporter metrics: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP045_WIDTH512_50K_TWO_PHASE_LR_RF1029\run_001\validation_metrics_report.csv
Reference reporter metrics: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP044_WIDTH256_25K_TO_50K_COSINE_LR\run_001\validation_metrics_report.csv
Card: D:\Projects\EPIC\EPIC\cnn_exp\CNN-EXP-045.md
SAVE_EXPERIMENT=False. Сначала пришли EXP045 result JSON и metrics.
